# Lesson LIN 3: Matrix Inverses, Conditioning, & Multicollinearity [SOLUTIONS GUIDE]

> **SOLUTIONS GUIDE.** Reference implementations with the same test cells the
> student workbook uses. Executed in CI by `tests/unit/test_curriculum.py`.
## Track 1: Linear Algebra for Neural Arrays

### Clinical & Biophysical Motivation
In electrophysiology, we frequently solve **linear inverse problems**:
1. **Neural Source Localization (EEG/MEG/ECoG)**: Given potentials $V$ measured on $C$ surface electrodes, estimate the current source amplitudes $s$ generated by $P$ brain sources. The forward model is $V = L s$, and $L$ is rectangular in general, so it cannot simply be inverted. That is the subject of this module.
2. **Optimal Spatial Filtering (Beamforming & BCI Decoders)**: Inverting the sensor covariance matrix $\Sigma^{-1}$ to isolate deep target signals (like subthalamic beta bursts) while nulling cortical interference.

However, high-density neural arrays (e.g. 384-channel Neuropixels shanks, 64-electrode ECoG grids, and directional DBS leads with $0.5\text{ mm}$ contact spacing) introduce a severe hazard: **Multicollinearity**.
When two electrodes are physically adjacent, volume conduction causes them to record virtually identical mixtures of neural activity. Their corresponding rows in the forward operator matrix become almost linearly dependent, driving the matrix **condition number** $\kappa(A)$ into the thousands.

In this module, you will build the mathematical foundations of matrix inversion, diagnose ill-conditioning from scratch via SVD, witness how $1\%$ recording noise creates a $380\%$ catastrophe under unregularized inversion, and implement **Tikhonov regularization** (Ridge Regression) to restore numerical stability.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Seed for reproducibility
np.random.seed(42)
print("Environment initialized for Lesson LIN 3!")


---
### STEP 1: The Geometry of the Condition Number $\kappa(A)$

A square matrix $A \in \mathbb{R}^{n \times n}$ has an inverse $A^{-1}$ if and only if its determinant is non-zero: $\det(A) \neq 0$.
Geometrically, $\det(A)$ represents the volume scaling factor of the linear transformation. If $\det(A) = 0$, $A$ crushes an $n$-dimensional volume onto a flat subspace of lower dimension, making inversion impossible.

In numerical computing, however, $\det(A) \neq 0$ is **not sufficient**. A matrix can have a non-zero determinant while still being completely unusable in practice due to floating-point sensitivity.
The true measure of matrix invertibility is the **Condition Number** $\kappa(A)$:
$$\kappa(A) = \|A\| \cdot \|A^{-1}\| = \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}$$
where $\sigma_{\max}$ and $\sigma_{\min}$ are the largest and smallest singular values from the **Singular Value Decomposition (SVD)** $A = U \Sigma V^T$.

- If $\kappa(A) \approx 1$: The matrix is **well-conditioned** (e.g. orthogonal rotation). It stretches all state-space axes equally. A unit hypersphere transforms into a sphere.
- If $\kappa(A) > 10^3$: The matrix is **ill-conditioned**. It stretches one axis enormously while flattening another into a razor-thin pancake.

**Task 1**: Implement `compute_condition_number(A: np.ndarray) -> float` using `np.linalg.svd`.
Do not use `np.linalg.cond`.

In [ ]:
def compute_condition_number(A: np.ndarray) -> float:
    """Compute the 2-norm condition number of matrix A via SVD.
    
    Returns:
        float: sigma_max / sigma_min. Returns np.inf if the smallest singular value is 0.
    """
    s = np.linalg.svd(A, compute_uv=False)
    if s[-1] < 1e-15:
        return float(np.inf)
    return float(s[0] / s[-1])


In [ ]:
# --- TEST CELL FOR STEP 1 ---
# 1. Test orthogonal matrix (kappa must be exactly 1.0)
theta = np.pi / 3
Q = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
k_Q = compute_condition_number(Q)
assert np.isclose(k_Q, 1.0), f'Orthogonal matrix must have kappa=1.0, got {k_Q}'

# 2. Test singular matrix (kappa must be infinity)
A_sing = np.array([[1.0, 2.0], [2.0, 4.0]])
assert np.isinf(compute_condition_number(A_sing)), 'Singular matrix must have infinite condition number'

# 3. Test stretched matrix
A_diag = np.diag([1000.0, 1.0])
assert np.isclose(compute_condition_number(A_diag), 1000.0), 'Expected kappa=1000.0'
print('✅ Step 1 Passed! Condition number accurately extracted from SVD spectrum.')


---
### STEP 2: The Forward Biophysical Model (Leadfield Matrix $L$)

Consider $P=3$ point current sources located inside a deep brain structure (e.g. STN or motor thalamus):
$$\mathbf{r}_{\text{source}, 1} = [-1, 0, 0], \quad \mathbf{r}_{\text{source}, 2} = [0, 0, 0], \quad \mathbf{r}_{\text{source}, 3} = [1, 0, 0] \quad (\text{in mm})$$

We record extracellular potentials on $C=4$ electrode contacts placed at distance $y = 2.0\text{ mm}$.
Assuming a homogeneous, isotropic brain tissue volume conductor with conductivity $\sigma = 0.3\ \text{S/m}$, the potential $V_i$ at electrode $\mathbf{r}_{\text{elec}, i}$ due to current source $s_j$ at $\mathbf{r}_{\text{src}, j}$ is given by the quasi-static monopole equation:
$$L_{ij} = \frac{1}{4\pi \sigma \|\mathbf{r}_{\text{elec}, i} - \mathbf{r}_{\text{src}, j}\|}$$

The vector of measured potentials is given by the forward matrix-vector product:
$$V = L s \in \mathbb{R}^C$$

**Be precise about what this model is.** $1/(4\pi\sigma r)$ is the potential of a **point current source**, a monopole, and $s_j$ is a current in amperes. It is *not* a dipole. A current dipole falls off as $1/r^2$, carries a direction, and has units of ampere-metres:
$$V_{\text{dipole}} = \frac{\mathbf{p} \cdot \hat{\mathbf{r}}}{4\pi\sigma \|\mathbf{r}\|^2}$$
Real EEG and MEG leadfields are dipolar, because a neuron's source and sink are separated by a small distance and the monopole terms largely cancel at a distance. The monopole kernel is used here because it is the simplest thing that produces a genuinely ill-conditioned $L$, which is what this module is about. Carry the conditioning lesson forward. Do not carry this kernel into a real source-localization problem.

**Task 2**: Implement `build_leadfield_matrix(electrode_pos, source_pos, conductivity=0.3)` and compare the condition number $\kappa(L)$ between:
- **Wide array spacing** ($2.0\text{ mm}$ between contacts, e.g. macro-contacts).
- **Dense array spacing** ($0.1\text{ mm}$ between contacts, e.g. high-density directional DBS segments or Neuropixels).

In [ ]:
def build_leadfield_matrix(
    electrode_pos: np.ndarray, 
    source_pos: np.ndarray, 
    conductivity: float = 0.3
) -> np.ndarray:
    """Construct the [N_elec x N_src] leadfield matrix using extracellular volume conduction."""
    n_elec = len(electrode_pos)
    n_src = len(source_pos)
    L = np.zeros((n_elec, n_src), dtype=float)
    for i in range(n_elec):
        for j in range(n_src):
            r = np.linalg.norm(electrode_pos[i] - source_pos[j])
            L[i, j] = 1.0 / (4.0 * np.pi * conductivity * max(r, 1e-6))
    return L


In [ ]:
# --- TEST CELL FOR STEP 2 ---
sources = np.array([[-1.0, 0.0, 0.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])

# Wide electrode array (spacing 2.0 mm)
elec_wide = np.array([[-3.0, 2.0, 0.0], [-1.0, 2.0, 0.0], [1.0, 2.0, 0.0], [3.0, 2.0, 0.0]])
L_wide = build_leadfield_matrix(elec_wide, sources)
k_wide = compute_condition_number(L_wide)

# Dense electrode array (spacing 0.1 mm)
elec_dense = np.array([[-0.15, 2.0, 0.0], [-0.05, 2.0, 0.0], [0.05, 2.0, 0.0], [0.15, 2.0, 0.0]])
L_dense = build_leadfield_matrix(elec_dense, sources)
k_dense = compute_condition_number(L_dense)

print(f'Wide Array Condition Number:  kappa(L_wide)  = {k_wide:.1f}')
print(f'Dense Array Condition Number: kappa(L_dense) = {k_dense:.1f}')

assert L_wide.shape == (4, 3)
assert L_dense.shape == (4, 3)
assert k_wide < 100.0, f'Wide array should be well-conditioned (<100), got {k_wide}'
assert k_dense > 1000.0, f'Dense array must be ill-conditioned (>1000), got {k_dense}'
print('✅ Step 2 Passed! Physical electrode proximity proved to cause multicollinearity and ill-conditioning.')


---
### STEP 3: The Noise Catastrophe & Tikhonov Regularization

To reconstruct the unknown neural source currents $s$ from measured noisy potentials $V_{\text{noisy}} = L s + \eta$, the textbook Ordinary Least Squares (OLS) solution is:
$$\hat{s}_{\text{OLS}} = (L^T L)^{-1} L^T V_{\text{noisy}} = L^+ V_{\text{noisy}}$$

The error in the source estimate is:
$$\delta s = \hat{s} - s = (L^T L)^{-1} L^T \eta$$

Expanding in the singular value basis of $L$, the variance of the error explodes along the smallest singular value $\sigma_{\min}$:
$$\mathbb{E}[\|\delta s\|^2] = \sigma_\eta^2 \sum_{i=1}^P \frac{1}{\sigma_i^2} \ge \frac{\sigma_\eta^2}{\sigma_{\min}^2}$$
When $\sigma_{\min} \approx 1.75 \times 10^{-4}$, even tiny microvolt noise is amplified by a factor of $\frac{1}{1.75 \times 10^{-4}} > 5700$!

#### Tikhonov Regularization (Ridge Inversion)
We constrain the source energy $\|s\|^2$ by minimizing the penalized cost function:
$$\mathcal{J}(s) = \|V - L s\|^2 + \lambda \|s\|^2$$
The analytical minimizer is the **Tikhonov-regularized inverse**:
$$\hat{s}_{\text{ridge}} = (L^T L + \lambda I)^{-1} L^T V$$

The regularization parameter $\lambda > 0$ elevates the eigenvalues of $L^T L$, guaranteeing that the effective condition number remains safely bounded.

**Task 3**: Implement `solve_inverse_tikhonov(L, V, lambda_reg=0.0)`.

In [ ]:
def solve_inverse_tikhonov(
    L: np.ndarray, 
    V: np.ndarray, 
    lambda_reg: float = 0.0
) -> np.ndarray:
    """Solve the linear inverse problem V = L @ s using Tikhonov (Ridge) regularization.
    
    Args:
        L: [C x P] Leadfield forward matrix.
        V: [C] or [C x T] Observed extracellular potentials.
        lambda_reg: Ridge regularization parameter lambda. If 0.0, computes unregularized OLS.
        
    Returns:
        s_hat: [P] or [P x T] Estimated neural source currents.
    """
    n_src = L.shape[1]
    if lambda_reg <= 0.0:
        return np.linalg.pinv(L) @ V
    
    # Regularized normal equations: (L^T @ L + lambda * I) @ s = L^T @ V
    LTL = L.T @ L
    reg_matrix = LTL + lambda_reg * np.eye(n_src)
    return np.linalg.solve(reg_matrix, L.T @ V)


In [ ]:
# --- TEST CELL FOR STEP 3 ---
# Ground-truth neural currents (in nA)
true_s = np.array([10.0, 25.0, -15.0])

# Forward potential without noise on dense array
V_clean = L_dense @ true_s

# Add 1% measurement noise
np.random.seed(42)
noise_amp = 0.01 * np.linalg.norm(V_clean)
noise = np.random.normal(0, noise_amp / np.sqrt(len(V_clean)), size=V_clean.shape)
V_noisy = V_clean + noise

# 1. Unregularized OLS inversion
s_ols = solve_inverse_tikhonov(L_dense, V_noisy, lambda_reg=0.0)
err_ols = np.linalg.norm(s_ols - true_s) / np.linalg.norm(true_s)

# 2. Tikhonov regularized inversion
lambda_optimal = 1e-5
s_ridge = solve_inverse_tikhonov(L_dense, V_noisy, lambda_reg=lambda_optimal)
err_ridge = np.linalg.norm(s_ridge - true_s) / np.linalg.norm(true_s)

print(f'True Sources:         {true_s}')
print(f'OLS Estimate:         {np.round(s_ols, 2)} (Relative Error: {err_ols:.1%})')
print(f'Tikhonov Estimate:    {np.round(s_ridge, 2)} (Relative Error: {err_ridge:.1%})')

# Assert that OLS suffered the catastrophic noise amplification (>200% error)
assert err_ols > 2.0, f'Expected OLS error to exceed 200% due to ill-conditioning, got {err_ols:.2f}'
# Assert that Tikhonov regularization reduced the error by more than 3x
assert err_ridge < 1.0, f'Tikhonov error must be <100%, got {err_ridge:.2f}'
assert err_ridge < (err_ols / 3.0), 'Tikhonov must achieve at least 3x error reduction'
print('✅ Step 3 Passed! Tikhonov regularization successfully stabilized the ill-conditioned inverse problem.')


---
### STEP 4: Guardrail Demo: The L-Curve and Inverting Singular Montages

Below we visualize:
1. **The L-Curve**: The fundamental Pareto frontier between residual fitting error $\|V - L \hat{s}\|^2$ and solution energy $\|\hat{s}\|^2$.
2. **The CAR Inversion Trap**: demonstrating that `np.linalg.inv(cov_car)` on Common Average Referenced data does **not** raise, and silently returns amplified round-off instead, and how `np.linalg.pinv` handles the rank-deficient subspace honestly.


In [ ]:
# 1. Compute L-Curve over a range of regularization parameters
lambdas = np.logspace(-9, -2, 50)
residuals = []
source_norms = []

for lam in lambdas:
    s_est = solve_inverse_tikhonov(L_dense, V_noisy, lambda_reg=lam)
    residuals.append(np.linalg.norm(V_noisy - L_dense @ s_est))
    source_norms.append(np.linalg.norm(s_est))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Plot L-Curve
ax1.plot(residuals, source_norms, 'o-', color='darkviolet', lw=2, markersize=4)
opt_idx = np.argmin(np.abs(lambdas - 1e-5))
ax1.plot(residuals[opt_idx], source_norms[opt_idx], 'r*', markersize=14, label=r'Optimal $\lambda = 10^{-5}$')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_title('The L-Curve: Residual Error vs Solution Norm', fontsize=11, fontweight='bold')
ax1.set_xlabel(r'Residual Fit Error $\|V - L\hat{s}\|$', fontsize=10)
ax1.set_ylabel(r'Solution Energy $\|\hat{s}\|$', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot Source Reconstruction Comparison
x_indices = np.arange(len(true_s))
width = 0.25
ax2.bar(x_indices - width, true_s, width, label='Ground Truth s', color='black', alpha=0.8)
ax2.bar(x_indices, s_ols, width, label='OLS (Unregularized)', color='crimson', alpha=0.7)
ax2.bar(x_indices + width, s_ridge, width, label=r'Tikhonov ($\lambda=10^{-5}$)', color='teal', alpha=0.8)
ax2.set_xticks(x_indices)
ax2.set_xticklabels(['Source 1 (STN)', 'Source 2 (STN)', 'Source 3 (STN)'])
ax2.set_ylabel('Source Amplitude (nA)', fontsize=10)
ax2.set_title('Source Reconstruction Under 1% Measurement Noise', fontsize=11, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.close('all')

# 2. Demonstrate the CAR Inversion Trap
C = 8
M_car = np.eye(C) - (1.0 / C) * np.ones((C, C))
X_sim = np.random.randn(C, 500)
X_car = M_car @ X_sim
cov_car = np.cov(X_car)

print('--- SCIENTIFIC GUARDRAIL CHECK ---')
print(f'Covariance Matrix Rank: {np.linalg.matrix_rank(cov_car)} of {C}')
print(f'Covariance Determinant: {np.linalg.det(cov_car):.2e}')
try:
    inv_fail = np.linalg.inv(cov_car)
    print('⚠️ WARNING: np.linalg.inv did not raise an error, but condition number is:', np.linalg.cond(cov_car))
except np.linalg.LinAlgError as e:
    print(f'✅ Caught expected mathematical exception: {e}')

# The correct safe approach
cov_pinv = np.linalg.pinv(cov_car, rcond=1e-10)
print(f'✅ Safe pseudoinverse computed within rank-{C-1} signal subspace.')
